# KulitKu: 3 Frozen CNN × 3 ML Classifiers
Kode: GitHub. Dataset dan output: Google Drive. Mesin: Colab GPU untuk ekstraksi; CPU untuk classifier.
Dataset harus sudah tersedia sebagai empat folder, bukan unduh seluruh Kaggle. Bobot pretrained membutuhkan internet. Audit augmentasi wajib sebelum split. Setiap eksperimen disimpan terpisah dan dapat dilewati saat sudah selesai.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE=Path('/content/drive/MyDrive/KulitKu')
BASE.mkdir(parents=True,exist_ok=True)


## 1. Kode dari GitHub atau ZIP
Untuk repo publik gunakan GitHub. Untuk repo private/belum terbit, simpan ZIP paket ini di MyDrive/KulitKu/code/KulitKu_Monorepo.zip dan aktifkan USE_DRIVE_ZIP. Jangan menulis token GitHub di notebook.


In [ ]:
import subprocess,sys,os,shutil,zipfile,json
USE_DRIVE_ZIP=True
REPO_URL='https://github.com/Liex2810/kulitku.git'
CODE_ZIP=BASE/'code/KulitKu_Monorepo.zip'
WORK=Path('/content/kulitku')
if not WORK.exists():
    if USE_DRIVE_ZIP:
        if not CODE_ZIP.exists():raise FileNotFoundError(f'Upload paket repo ke {CODE_ZIP}')
        unpack=Path('/content/kulitku_unpack');unpack.mkdir(exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP) as archive:
            for entry in archive.infolist():
                if not (unpack/entry.filename).resolve().is_relative_to(unpack.resolve()):raise ValueError('Unsafe ZIP path')
            archive.extractall(unpack)
        roots=[p.parent for p in unpack.rglob('pyproject.toml') if (p.parent/'machine-learning').is_dir()]
        if len(roots)!=1:raise ValueError('Expected one KulitKu root')
        shutil.copytree(roots[0],WORK)
    else:subprocess.run(['git','clone',REPO_URL,str(WORK)],check=True)
os.chdir(WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[ml]'],check=True)
import torch
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU; aktifkan GPU untuk ekstraksi nyata')


## 2. Lokasi dan versi eksperimen
RAW_DRIVE menunjuk folder yang langsung berisi Eczema, Herpes Zoster, Normal, Ringworm (alias folder Kaggle juga didukung). Jika dataset berubah, pakai snapshot lokal dan versi manifest baru. Output disimpan di Drive. Notebook tidak membuat asumsi otomatis bahwa nama aug_x sudah pasti dapat dipetakan ke gambar asal.


In [ ]:
RAW_DRIVE=BASE/'datasets/raw'
DATA=Path('/content/kulitku_data_v1')
if not RAW_DRIVE.is_dir():raise FileNotFoundError(f'Dataset belum tersedia di {RAW_DRIVE}')
if not DATA.exists():shutil.copytree(RAW_DRIVE,DATA)
INVENTORY=BASE/'manifests/inventory-v1.csv'
REVIEWED=BASE/'manifests/reviewed-v1.csv'
SPLIT=BASE/'manifests/split-v1'
FEATURES=BASE/'features/v1'
BENCHMARK=BASE/'experiments/benchmark-v1'
EVALUATION=BASE/'experiments/final-test-v1'
EXPORT=BASE/'exports/skin-v1'
CONFIG=WORK/'machine-learning/configs/default.json'

def run(module,*args):
    subprocess.run([sys.executable,'-m',module,*map(str,args)],cwd=WORK,check=True)


In [ ]:
if not INVENTORY.exists():run('kulitku_ml.data','inventory','--data',DATA,'--output',INVENTORY)
import pandas as pd
from IPython.display import display,Image as DisplayImage
display(pd.read_csv(INVENTORY).head())
display(DisplayImage(filename=str(INVENTORY.parent/'eda_inventory/eda.png')))


## 3. Audit manual: wajib
Buat salinan inventory-v1.csv bernama reviewed-v1.csv. Isi group_id yang sama untuk pasien/gambar asal dan semua turunannya; is_original=1 untuk gambar asli terverifikasi, 0 untuk augmentasi; include=0 untuk gambar yang dikeluarkan. Catat alasannya di kolom tambahan audit_note. Jangan menandai semua gambar sebagai original hanya untuk meloloskan pipeline. Jika asal gambar tidak diketahui, cari sumber/metadata/original dataset terlebih dahulu. Pipeline sengaja berhenti jika informasi ini belum lengkap. phash_candidates.csv hanya kandidat hash sama; bukan jaminan lineage lengkap.


In [ ]:
APPROVE_AUDIT=False
if not APPROVE_AUDIT:raise RuntimeError('Selesaikan audit lalu set APPROVE_AUDIT=True')
if not REVIEWED.exists():raise FileNotFoundError(REVIEWED)
if not (SPLIT/'manifest.csv').exists():
    run('kulitku_ml.data','split','--manifest',REVIEWED,'--output',SPLIT,'--config',CONFIG,'--approve-audit')
display(pd.read_csv(SPLIT/'counts.csv'))


## 4. Ekstraksi frozen CNN
Bobot ImageNet diunduh, lalu disimpan bersama fitur ke Drive. Fitur/progress disimpan per batch; menjalankan sel ulang melanjutkan batch terakhir selesai. Tidak ada fine-tuning. Validation/test memakai preprocessing deterministik yang sama dengan backend. Tidak ada gambar test augmentasi yang disertakan.


In [ ]:
cfg=json.loads(CONFIG.read_text())
for key in cfg['models']:
    run('kulitku_ml.features','--data',DATA,'--manifest',SPLIT/'manifest.csv','--config',CONFIG,'--model',key,'--output',FEATURES/key)


## 5. Benchmark 9 kombinasi dengan grouped folds
Training fold memakai originals+augmentasi kelompok training. Validation fold memakai originals kelompok held-out. Scaler difit di dalam Pipeline setiap fold. Classifier selesai disimpan dan dilewati saat rerun, tetapi pencarian yang terputus di tengah satu kombinasi diulang. SVM kernel RBF dapat lama pada fitur besar; sesuaikan grid SEBELUM ekstraksi/eksperimen, bukan setelah melihat test.


In [ ]:
run('kulitku_ml.benchmark','run','--features',FEATURES,'--config',CONFIG,'--output',BENCHMARK)
display(pd.read_csv(BENCHMARK/'leaderboard.csv')[['cnn','classifier','cv_macro_f1_mean','cv_macro_f1_std']])
print((BENCHMARK/'winner.json').read_text())


In [ ]:
# Optional: CPU comparison includes preprocessing+CNN+scaler+classifier, using development originals.
RUN_PROFILE=False
if RUN_PROFILE:run('kulitku_ml.profile','--benchmark',BENCHMARK,'--data',DATA)


## 6. Final test setelah keputusan dikunci
Pemenang dipilih dari CV, kemudian otomatis refit pada seluruh development oleh GridSearchCV. Final test hanya menilai pemenang, bukan memilih ulang dari 9 model. Tidak ada overwrite laporan final. Skor CV tuning bukan estimasi independen setara final test. External smartphone test perlu data/label independen tambahan.


In [ ]:
if not EVALUATION.exists():run('kulitku_ml.benchmark','evaluate','--benchmark',BENCHMARK,'--output',EVALUATION)
print((EVALUATION/'metrics.json').read_text())
display(DisplayImage(filename=str(EVALUATION/'confusion_matrix.png')))


## 7. Ekspor untuk FastAPI
CNN diekspor ke ONNX, classifier+scaler disimpan sebagai sklearn Pipeline joblib. Backend tidak perlu PyTorch. Gunakan hanya artefak joblib milikmu yang dipercaya. Ekspor memeriksa kesetaraan fitur dan kelas. Download seluruh bundle; bukan classifier saja.


In [ ]:
if not (EXPORT/'checksums.json').exists():
    run('kulitku_ml.export','--benchmark',BENCHMARK,'--output',EXPORT,'--version','skin-v1','--data',DATA)
archive=shutil.make_archive(str(EXPORT)+'-bundle','zip',root_dir=EXPORT)
print('Paket backend:',archive)
